In [1]:
import findspark
findspark.init()

from pyspark.conf import SparkConf
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

conf = SparkConf().setAppName("1204").setMaster("local[4]")
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/07 01:24:47 WARN Utils: Your hostname, de24, resolves to a loopback address: 127.0.1.1; using 192.168.0.103 instead (on interface enp0s3)
25/08/07 01:24:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/07 01:24:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/08/07 01:24:50 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [ ]:
'''
Table: Queue

+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| person_id   | int     |
| person_name | varchar |
| weight      | int     |
| turn        | int     |
+-------------+---------+
person_id column contains unique values.
This table has the information about all people waiting for a bus.
The person_id and turn columns will contain all numbers from 1 to n, 
where n is the number of rows in the table.
turn determines the order of which the people will board the bus, 
where turn=1 denotes the first person to board and turn=n denotes the last person to board.
weight is the weight of the person in kilograms.
 

There is a queue of people waiting to board a bus. However, 
the bus has a weight limit of 1000 kilograms, so there may be some people who cannot board.

Write a solution to find the person_name of the last person that can fit on the bus without 
exceeding the weight limit. 
The test cases are generated such that the first person does not exceed the weight limit.

The result format is in the following example.

 

Example 1:

Input: 
Queue table:
+-----------+-------------+--------+------+
| person_id | person_name | weight | turn |
+-----------+-------------+--------+------+
| 5         | Alice       | 250    | 1    |
| 4         | Bob         | 175    | 5    |
| 3         | Alex        | 350    | 2    |
| 6         | John Cena   | 400    | 3    |
| 1         | Winston     | 500    | 6    |
| 2         | Marie       | 200    | 4    |
+-----------+-------------+--------+------+
Output: 
+-------------+
| person_name |
+-------------+
| John Cena   |
+-------------+
Explanation: The folowing table is ordered by the turn for simplicity.
+------+----+-----------+--------+--------------+
| Turn | ID | Name      | Weight | Total Weight |
+------+----+-----------+--------+--------------+
| 1    | 5  | Alice     | 250    | 250          |
| 2    | 3  | Alex      | 350    | 600          |
| 3    | 6  | John Cena | 400    | 1000         | (last person to board)
| 4    | 2  | Marie     | 200    | 1200         | (cannot board)
| 5    | 4  | Bob       | 175    | ___          |
| 6    | 1  | Winston   | 500    | ___          |
+------+----+-----------+--------+--------------+
'''

In [2]:
data = [
(5,'Alice'     ,250,1),
(4,'Bob'       ,175,5),
(3,'Alex'      ,350,2),
(6,'John Cena' ,400,3),
(1,'Winston'   ,500,6),
(2,'Marie'     ,200,4)   
]
schema = ['person_id','person_name','weight','turn']

In [3]:
df = spark.createDataFrame(data = data, schema= schema)
df.show()

+---------+-----------+------+----+
|person_id|person_name|weight|turn|
+---------+-----------+------+----+
|        5|      Alice|   250|   1|
|        4|        Bob|   175|   5|
|        3|       Alex|   350|   2|
|        6|  John Cena|   400|   3|
|        1|    Winston|   500|   6|
|        2|      Marie|   200|   4|
+---------+-----------+------+----+



In [4]:
from pyspark.sql.window import Window

In [13]:
windows_spec = Window.orderBy(F.col("turn"))
df_weightsum = df.withColumn('weightSum', F.sum(F.col("weight")).over(windows_spec))

df_weightsum.show()

# df_weightsum.where(F.col("weightSum") <= 1000)\
#             .select(F.max(F.col("weightSum")))

df_weightsum.where(F.col("weightSum") == (df_weightsum.where(F.col("weightSum") <= 1000)\
            .select(F.max(F.col("weightSum"))).collect()[0][0]
                                         )
                  )\
            .select(F.col("person_name"))\
            .show()

25/08/07 01:49:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/07 01:49:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/07 01:49:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/07 01:49:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/07 01:49:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---------+-----------+------+----+---------+
|person_id|person_name|weight|turn|weightSum|
+---------+-----------+------+----+---------+
|        5|      Alice|   250|   1|      250|
|        3|       Alex|   350|   2|      600|
|        6|  John Cena|   400|   3|     1000|
|        2|      Marie|   200|   4|     1200|
|        4|        Bob|   175|   5|     1375|
|        1|    Winston|   500|   6|     1875|
+---------+-----------+------+----+---------+



25/08/07 01:49:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/07 01:49:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/07 01:49:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/07 01:49:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/07 01:49:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/07 01:49:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/07 0

+-----------+
|person_name|
+-----------+
|  John Cena|
+-----------+



25/08/07 01:49:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/08/07 01:49:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


## Melwin You need to understand this method 
## Very much required

## SQL Solution
<pre>
WITH TEMP AS (
    SELECT person_id,person_name,weight,turn,
           SUM(weight) OVER(ORDER BY turn) as sum_weight
    FROM Queue
)
SELECT person_name FROM 
TEMP WHERE sum_weight = (
SELECT MAX(sum_weight) FROM TEMP 
WHERE sum_weight <= 1000 )
</pre>